<a href="https://colab.research.google.com/github/alisony755/DS4400/blob/main/HW4/DS3000_HW4_Problem4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Problem 4

In [141]:
# Import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.preprocessing import OrdinalEncoder

# Package Naive Bayes
from sklearn.naive_bayes import CategoricalNB

In [142]:
# Get mushroom data
!pip install -U ucimlrepo

from ucimlrepo import fetch_ucirepo

# fetch dataset
mushroom = fetch_ucirepo(id=73)

# data (as pandas dataframes)
X = mushroom.data.features
y = mushroom.data.targets

# metadata
print(mushroom.metadata)

# variable information
print(mushroom.variables)

{'uci_id': 73, 'name': 'Mushroom', 'repository_url': 'https://archive.ics.uci.edu/dataset/73/mushroom', 'data_url': 'https://archive.ics.uci.edu/static/public/73/data.csv', 'abstract': 'From Audobon Society Field Guide; mushrooms described in terms of physical characteristics; classification: poisonous or edible', 'area': 'Biology', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 8124, 'num_features': 22, 'feature_types': ['Categorical'], 'demographics': [], 'target_col': ['poisonous'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 1981, 'last_updated': 'Thu Aug 10 2023', 'dataset_doi': '10.24432/C5959T', 'creators': [], 'intro_paper': None, 'additional_info': {'summary': "This data set includes descriptions of hypothetical samples corresponding to 23 species of gilled mushrooms in the Agaricus and Lepiota Family (pp. 500-525).  Each species is identified as definitely edible, definitely po

### 4.1

In [143]:
X = mushroom.data.features
y = mushroom.data.targets.values.ravel()

# Encode features
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X)

In [144]:
# Split dataset into training (75%) and testing (25%)
X_train_df, X_test_df, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [145]:
def train_naive_bayes(X_train, y_train):
  """ Computes prior and conditional probabilities for Naive Bayes classifier.

  Args:
      X_train (pd.DataFrame): Training features
      y_train (pd.Series or np.ndarray): Training labels (Edible / Poisonous)

  Returns:
      dict: Dictionary containing priors and likelihoods
  """

  model = {}  # Store learned probabilities

  # Get class labels
  classes = np.unique(y_train)

  # Compute prior probabilities P(Y)
  priors = {}
  for c in classes:
      priors[c] = np.sum(y_train == c) / len(y_train)

  model["priors"] = priors

  # Conditional probabilities P(X_i | Y)
  likelihoods = {}

  # Loop over each feature column
  for col in X_train.columns:
      likelihoods[col] = {}

      # All possible values of this categorical feature
      feature_values = X_train[col].unique()

      # Compute probability table for each class
      for c in classes:
          subset = X_train[y_train == c]  # filter rows of class c

          value_probs = {}

          # Laplace smoothing denominator (add number of feature values)
          denom = len(subset) + len(feature_values)

          # Compute P(X_i = x | Y = c)
          for v in feature_values:
              count = np.sum(subset[col] == v)
              value_probs[v] = (count + 1) / denom  # Laplace smoothing

          likelihoods[col][c] = value_probs

  model["likelihoods"] = likelihoods

  return model

In [146]:
def predict_naive_bayes(model, X_data, dataset_name="Dataset"):
  """ Predicts class labels and prints Naive Bayes probabilities.

  Args:
      model (dict): Trained Naive Bayes model (priors + likelihoods)
      X_data (pd.DataFrame): Feature data (training or testing)
      dataset_name (str): Name of dataset ("Training" or "Testing")

  Returns:
      np.ndarray: Predicted class labels
  """

  print(f"\n{dataset_name}")

  # Print prior probabilities
  print("\nPrior Probabilities:")
  for cls, prob in model["priors"].items():
      label = "Edible" if cls == 'e' else "Poisonous"
      print(f"P(Y = {label}) = {prob:.4f}")

  # Print first few conditional probabilities
  print("\nConditional Probabilities:")

  # Limit to first 5 features
  for feature in list(model["likelihoods"].keys())[:5]:
      print(f"\nFeature: {feature}")

      for cls in model["likelihoods"][feature]:
          label = "Edible" if cls == 'e' else "Poisonous"
          print(f"  Class: {label}")

          for val, prob in model["likelihoods"][feature][cls].items():
              print(f"    P({feature}={val} | Y={label}) = {prob:.4f}")

  predictions = []

  # Compute posterior probabilities
  for i, (_, row) in enumerate(X_data.iterrows()):

      class_probs = {}

      for c in model["priors"]:
          prob = model["priors"][c]

          for col in X_data.columns:
              val = row[col]
              prob *= model["likelihoods"][col][c].get(val, 1e-6)

          class_probs[c] = prob

      # Normalize probabilities
      total = sum(class_probs.values())
      for c in class_probs:
          class_probs[c] /= total

      edible_prob = class_probs.get('e', 0)
      poison_prob = class_probs.get('p', 0)

      # Print first 5 samples
      if i < 5:
          print(f"\nSample {i+1}:")
          print(f"P(Edible | X)    = {edible_prob:.6f}")
          print(f"P(Poisonous | X) = {poison_prob:.6f}")

      predictions.append(max(class_probs, key=class_probs.get))

  return np.array(predictions)

In [147]:
# Train Naive Bayes model
nb_model = train_naive_bayes(X_train_df, y_train)
y_pred_train = predict_naive_bayes(nb_model, X_train_df, "Training Data")


Training Data

Prior Probabilities:
P(Y = Edible) = 0.5180
P(Y = Poisonous) = 0.4820

Conditional Probabilities:

Feature: cap-shape
  Class: Edible
    P(cap-shape=x | Y=Edible) = 0.4684
    P(cap-shape=k | Y=Edible) = 0.0547
    P(cap-shape=f | Y=Edible) = 0.3751
    P(cap-shape=b | Y=Edible) = 0.0936
    P(cap-shape=s | Y=Edible) = 0.0079
    P(cap-shape=c | Y=Edible) = 0.0003
  Class: Poisonous
    P(cap-shape=x | Y=Poisonous) = 0.4410
    P(cap-shape=k | Y=Poisonous) = 0.1485
    P(cap-shape=f | Y=Poisonous) = 0.3955
    P(cap-shape=b | Y=Poisonous) = 0.0129
    P(cap-shape=s | Y=Poisonous) = 0.0003
    P(cap-shape=c | Y=Poisonous) = 0.0017

Feature: cap-surface
  Class: Edible
    P(cap-surface=s | Y=Edible) = 0.2766
    P(cap-surface=f | Y=Edible) = 0.3722
    P(cap-surface=y | Y=Edible) = 0.3509
    P(cap-surface=g | Y=Edible) = 0.0003
  Class: Poisonous
    P(cap-surface=s | Y=Poisonous) = 0.3611
    P(cap-surface=f | Y=Poisonous) = 0.2016
    P(cap-surface=y | Y=Poisonous) =

### 4.2

In [148]:
# Encode X for model
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X)

X_train_enc, X_test_enc, y_train_enc, y_test_enc = train_test_split(
    X_encoded, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

In [149]:
# Generate predictions on test set
y_pred_test = predict_naive_bayes(nb_model, X_test_df, "Testing Data")


Testing Data

Prior Probabilities:
P(Y = Edible) = 0.5180
P(Y = Poisonous) = 0.4820

Conditional Probabilities:

Feature: cap-shape
  Class: Edible
    P(cap-shape=x | Y=Edible) = 0.4684
    P(cap-shape=k | Y=Edible) = 0.0547
    P(cap-shape=f | Y=Edible) = 0.3751
    P(cap-shape=b | Y=Edible) = 0.0936
    P(cap-shape=s | Y=Edible) = 0.0079
    P(cap-shape=c | Y=Edible) = 0.0003
  Class: Poisonous
    P(cap-shape=x | Y=Poisonous) = 0.4410
    P(cap-shape=k | Y=Poisonous) = 0.1485
    P(cap-shape=f | Y=Poisonous) = 0.3955
    P(cap-shape=b | Y=Poisonous) = 0.0129
    P(cap-shape=s | Y=Poisonous) = 0.0003
    P(cap-shape=c | Y=Poisonous) = 0.0017

Feature: cap-surface
  Class: Edible
    P(cap-surface=s | Y=Edible) = 0.2766
    P(cap-surface=f | Y=Edible) = 0.3722
    P(cap-surface=y | Y=Edible) = 0.3509
    P(cap-surface=g | Y=Edible) = 0.0003
  Class: Poisonous
    P(cap-surface=s | Y=Poisonous) = 0.3611
    P(cap-surface=f | Y=Poisonous) = 0.2016
    P(cap-surface=y | Y=Poisonous) = 

### 4.3

In [150]:
def compute_metrics_nb(y_true, y_pred):
    """ Computes classification metrics for Naive Bayes model.

    Args:
        y_true (np.ndarray): True labels
        y_pred (np.ndarray): Predicted labels

    Returns:
        tuple: (accuracy, precision, recall, f1)
    """

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, pos_label='p')
    recall = recall_score(y_true, y_pred, pos_label='p')
    f1 = f1_score(y_true, y_pred, pos_label='p')

    print("\nNaive Bayes Results:")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    return accuracy, precision, recall, f1

In [151]:
# Evaluate custom Naive Bayes model
y_pred_nb = predict_naive_bayes(nb_model, X_test_df, "Testing Data")
custom_metrics = compute_metrics_nb(y_test, y_pred_nb)


Testing Data

Prior Probabilities:
P(Y = Edible) = 0.5180
P(Y = Poisonous) = 0.4820

Conditional Probabilities:

Feature: cap-shape
  Class: Edible
    P(cap-shape=x | Y=Edible) = 0.4684
    P(cap-shape=k | Y=Edible) = 0.0547
    P(cap-shape=f | Y=Edible) = 0.3751
    P(cap-shape=b | Y=Edible) = 0.0936
    P(cap-shape=s | Y=Edible) = 0.0079
    P(cap-shape=c | Y=Edible) = 0.0003
  Class: Poisonous
    P(cap-shape=x | Y=Poisonous) = 0.4410
    P(cap-shape=k | Y=Poisonous) = 0.1485
    P(cap-shape=f | Y=Poisonous) = 0.3955
    P(cap-shape=b | Y=Poisonous) = 0.0129
    P(cap-shape=s | Y=Poisonous) = 0.0003
    P(cap-shape=c | Y=Poisonous) = 0.0017

Feature: cap-surface
  Class: Edible
    P(cap-surface=s | Y=Edible) = 0.2766
    P(cap-surface=f | Y=Edible) = 0.3722
    P(cap-surface=y | Y=Edible) = 0.3509
    P(cap-surface=g | Y=Edible) = 0.0003
  Class: Poisonous
    P(cap-surface=s | Y=Poisonous) = 0.3611
    P(cap-surface=f | Y=Poisonous) = 0.2016
    P(cap-surface=y | Y=Poisonous) = 

### 4.4

In [152]:
# Fill missing values with "missing"
X = X.fillna("missing")

In [153]:
# Encode categorical features into integers
encoder = OrdinalEncoder()
X_encoded = encoder.fit_transform(X)

# Train/test split
X_train_enc, X_test_enc, y_train_enc, y_test_enc = train_test_split(
    X_encoded, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

# Train Naive Bayes model
sk_nb = CategoricalNB()
sk_nb.fit(X_train_enc, y_train_enc)

# Predict
y_pred_sk = sk_nb.predict(X_test_enc)

In [154]:
# Evaluate sklearn model
sk_metrics = compute_metrics_nb(y_test, y_pred_sk)


Naive Bayes Results:
Accuracy: 0.9527
Precision: 0.9911
Recall: 0.9101
F1 Score: 0.9489
